<a href="https://colab.research.google.com/github/eojin22/ESAA/blob/main/OB_WEEK01_1.%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**0904 수상작 리뷰**



# 주제 및 데이터

### **주제**: **의류 제조 회사 생산성 예측 AI 해커톤**

### **데이터 구성:**

- **train.csv**
    - 1,197개의 데이터
    - `ID` : 샘플별 고유 ID
    - 요일, 부서, 작업환경 등 의류 제조 공정과 관련된 정보
    - `actual_productivity` : 실제 생산성(생산량)
- **test.csv**
    - 818개의 데이터
    - `ID` : 샘플별 고유 ID
    - 요일, 부서, 작업환경 등의 정보
    - `actual_productivity`는 제공되지 않으며 이를 예측하는 것이 목표
- **column_info.csv**
    - 데이터의 각 컬럼에 대한 추가 설명
- **sample_submission.csv**
    - `ID` : 샘플별 고유 ID
    - `actual_productivity` : 예측한 생산성 값



---



# 코드 리뷰

1. 데이터 확인 및 전처리
- 칼럼 정보, 중복값, 결측치 확인
- 이상치는 데이터 수가 적어 제거하지 않음
- Train 데이터의 평균을 이용해 결측치 처리
- Skewness가 0.75 이상인 변수에 로그 변환 적용
- Target 변수에도 로그 변환 적용
필요한 변수 Scaling

2. 모델 학습 및 성능 비교
- RF, Decision Tree, GBR, ETR, XGB, LGBM, ElasticNet, AdaBoost, CAT, NGB, HistGBR, LR, Lasso, Ridge 등 다양한 모델 학습
- 대회 평가 지표인 NMAE를 기준으로 모델 성능 비교
- 개별 모델 성능은 CAT > GBR > NGB > XGB > ETR > RF 순으로 나타남

3. K-Fold 검증
- K-Fold 적용 후 성능이 오히려 감소하는 것을 확인-> 별도의 K-Fold 기반 튜닝을 진행하지 않고 기존 모델 성능을 활용

4. 앙상블
-  성능이 좋은 모델 8개를 선정-> LGBM, HistGBR을 제외한 모델을 활용
-  Soft Voting 방식으로 여러 모델의 예측값을 결합 후 최종 예측값을 생성

In [ ]:
def NMAE(y_true, y_pred):
    return np.mean(
        np.abs(y_true - y_pred)
    ) / np.mean(np.abs(y_true))

# 낮을수록 좋은 평가 지표로 설정
nmae_scorer = make_scorer(
    NMAE,
    greater_is_better=False
)

In [ ]:
models = {
    'RF': RandomForestRegressor(...),
    'GBR': GradientBoostingRegressor(...),
    'ETR': ExtraTreesRegressor(...),
    'XGB': XGBRegressor(...),
    'CAT': CatBoostRegressor(...),
    'NGB': NGBRegressor(...)
}

scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)

    scores[name] = NMAE(y_valid, pred)



---



# 차별점 및 배울 점

**다양한 모델을 직접 비교**

- 처음부터 특정 모델 하나를 선택하기보다 Random Forest, GBR, ETR, XGB, LGBM, CAT, NGB, Linear Regression, Lasso, Ridge 등 **다양한 회귀 모델을 직접 학습하고 성능을 비교**하였다. 그 결과 **CAT > GBR > NGB > XGB > ETR > RF** 순으로 성능 차이를 확인하고, 좋은 성능을 보이는 모델을 중심으로 최종 앙상블을 구성했다는 점이 인상적이었다.

**모델의 다양성과 앙상블 성능**
   - Random Forest와 Extra Trees처럼 비슷한 계열의 모델뿐만 아니라 Gradient Boosting, XGBoost, NGBoost, CatBoost 등 서로 다른 Boosting 계열 모델까지 함께 사용했다. 따라서 단순히 모델 개수를 늘리는 것이 아니라 서로 다른 모델이 만들어내는 예측의 차이를 활용하는 것이 중요하다는 것을 알 수 있었다.
    
**정형 데이터에서 트리 기반 모델의 강점**
- 의류 제조 데이터처럼 요일, 부서, 작업환경 등 다양한 형태의 변수가 존재하는 정형 데이터에서는 트리 기반 모델이 효과적으로 활용될 수 있음을 알게 되었다.